# 01 · Exploración de la API REData

Objetivo: entender cómo funciona la API de Red Eléctrica antes de escribir el script de extracción.

Empezamos con la serie más sencilla: **demanda diaria de una semana**.

## 1. Importar librerías

In [13]:
import requests       # para hacer peticiones HTTP a la API
import pandas as pd   # para trabajar con tablas (DataFrames)

## 2. Preparar la petición

La URL tiene la forma: `https://apidatos.ree.es/{idioma}/datos/{categoria}/{widget}`

Los parámetros (fechas y agrupación temporal) se pasan aparte en un diccionario.

In [14]:
URL_BASE = "https://apidatos.ree.es/es/datos"

categoria = "demanda"
widget = "evolucion"
url = f"{URL_BASE}/{categoria}/{widget}"

parametros = {
    "start_date": "2025-01-01T00:00",
    "end_date": "2025-01-07T23:59",
    "time_trunc": "day",   # agrupación: hour, day, month o year
}

print(url)

https://apidatos.ree.es/es/datos/demanda/evolucion


## 3. Hacer la petición y comprobar que ha ido bien

- `status_code` 200 significa que todo ha ido bien.
- `respuesta.url` muestra la URL completa que ha montado `requests` con los parámetros.

In [15]:
respuesta = requests.get(url, params=parametros, timeout=30)

print("Código de estado:", respuesta.status_code)
print("URL completa:", respuesta.url)

Código de estado: 200
URL completa: https://apidatos.ree.es/es/datos/demanda/evolucion?start_date=2025-01-01T00%3A00&end_date=2025-01-07T23%3A59&time_trunc=day


## 4. Explorar el JSON paso a paso

`respuesta.json()` convierte el texto de la respuesta en diccionarios y listas de Python.
Vamos bajando nivel a nivel hasta llegar a los datos.

In [16]:
datos = respuesta.json()

# Claves del primer nivel
print(datos.keys())

dict_keys(['data', 'included'])


In [17]:
# 'included' es una lista de series: ¿cuántas hay?
series = datos["included"]
print("Número de series:", len(series))

# Título de cada serie
for serie in series:
    print("-", serie["attributes"]["title"])

Número de series: 1
- Demanda


In [18]:
# Dentro de cada serie, los datos están en attributes -> values
valores = series[0]["attributes"]["values"]

print("Número de valores:", len(valores))
valores[:2]   # mostramos solo los dos primeros

Número de valores: 7


[{'value': 585942.98,
  'percentage': 1,
  'datetime': '2025-01-01T00:00:00.000+01:00'},
 {'value': 714746.165,
  'percentage': 1,
  'datetime': '2025-01-02T00:00:00.000+01:00'}]

## 5. Convertir los valores en una tabla de pandas

In [19]:
df = pd.DataFrame(valores)
df

,value,percentage,datetime
0,585942.980,1,2025-01-01T00:00:00.000+01:00
1,714746.165,1,2025-01-02T00:00:00.000+01:00
2,743131.902,1,2025-01-03T00:00:00.000+01:00
3,672654.205,1,2025-01-04T00:00:00.000+01:00
4,637473.016,1,2025-01-05T00:00:00.000+01:00
5,612375.246,1,2025-01-06T00:00:00.000+01:00
6,761126.956,1,2025-01-07T00:00:00.000+01:00


In [20]:
# Tipo de dato de cada columna
df.dtypes

value         float64
percentage      int64
datetime          str
dtype: object

## 6. Generación por tecnología: varias series

Misma semana y misma agrupación diaria, pero ahora pedimos la estructura de la generación.
Esta vez `included` trae **una serie por tecnología**.

In [21]:
categoria = "generacion"
widget = "estructura-generacion"
url = f"{URL_BASE}/{categoria}/{widget}"

# Reutilizamos el mismo diccionario de parámetros de la sección 2
respuesta = requests.get(url, params=parametros, timeout=30)
print("Código de estado:", respuesta.status_code)

series = respuesta.json()["included"]
print("Número de series:", len(series))

Código de estado: 200
Número de series: 16


In [22]:
# Nombre y tipo de cada serie
for serie in series:
    print(serie["attributes"]["title"], "->", serie["attributes"]["type"])

Hidráulica -> Renovable
Nuclear -> No-Renovable
Carbón -> No-Renovable
Motores diésel -> No-Renovable
Turbina de gas -> No-Renovable
Turbina de vapor -> No-Renovable
Ciclo combinado -> No-Renovable
Hidroeólica -> Renovable
Eólica -> Renovable
Solar fotovoltaica -> Renovable
Solar térmica -> Renovable
Otras renovables -> Renovable
Cogeneración -> No-Renovable
Residuos no renovables -> No-Renovable
Residuos renovables -> Renovable
Generación total -> total


### Juntar todas las series en una sola tabla

Para cada serie convertimos sus valores en un DataFrame (como en la sección 5)
y le añadimos dos columnas para saber de dónde viene cada fila: la tecnología y su tipo.
Al final unimos todas las tablas, una debajo de otra, con `pd.concat`.

In [23]:
tablas = []   # aquí guardaremos una tabla por serie

for serie in series:
    tabla = pd.DataFrame(serie["attributes"]["values"])
    tabla["tecnologia"] = serie["attributes"]["title"]
    tabla["tipo"] = serie["attributes"]["type"]
    tablas.append(tabla)

# Unimos todas las tablas; ignore_index=True renumera las filas desde 0
df_generacion = pd.concat(tablas, ignore_index=True)

print("Filas y columnas:", df_generacion.shape)
df_generacion.head(10)

Filas y columnas: (112, 5)


,value,percentage,datetime,tecnologia,tipo
0,58263.036,0.095216,2025-01-01T00:00:00.000+01:00,Hidráulica,Renovable
1,73779.807,0.100646,2025-01-02T00:00:00.000+01:00,Hidráulica,Renovable
2,70727.921,0.089678,2025-01-03T00:00:00.000+01:00,Hidráulica,Renovable
3,79809.182,0.108765,2025-01-04T00:00:00.000+01:00,Hidráulica,Renovable
4,45976.291,0.064850,2025-01-05T00:00:00.000+01:00,Hidráulica,Renovable
5,53733.847,0.076323,2025-01-06T00:00:00.000+01:00,Hidráulica,Renovable
6,71613.379,0.089334,2025-01-07T00:00:00.000+01:00,Hidráulica,Renovable
7,170905.239,0.279302,2025-01-01T00:00:00.000+01:00,Nuclear,No-Renovable
8,170892.777,0.233122,2025-01-02T00:00:00.000+01:00,Nuclear,No-Renovable
9,170647.412,0.216368,2025-01-03T00:00:00.000+01:00,Nuclear,No-Renovable


### Comprobación: ¿suman 1 los porcentajes de cada día?

`groupby("datetime")` agrupa las filas por día y `["percentage"].sum()` suma los porcentajes de cada grupo.

In [24]:
df_generacion.groupby("datetime")["percentage"].sum()

datetime
2025-01-01T00:00:00.000+01:00    2.0
2025-01-02T00:00:00.000+01:00    2.0
2025-01-03T00:00:00.000+01:00    2.0
2025-01-04T00:00:00.000+01:00    2.0
2025-01-05T00:00:00.000+01:00    2.0
2025-01-06T00:00:00.000+01:00    2.0
2025-01-07T00:00:00.000+01:00    2.0
Name: percentage, dtype: float64

## 7. Precios por hora

Pedimos los precios de un solo día con `time_trunc = "hour"`.

In [25]:
categoria = "mercados"
widget = "precios-mercados-tiempo-real"
url = f"{URL_BASE}/{categoria}/{widget}"

parametros_precios = {
    "start_date": "2025-01-15T00:00",
    "end_date": "2025-01-15T23:59",
    "time_trunc": "hour",
}

respuesta = requests.get(url, params=parametros_precios, timeout=30)
print("Código de estado:", respuesta.status_code)

series = respuesta.json()["included"]

# Nombre de cada serie y cuántos valores trae para un solo día
for serie in series:
    print(serie["attributes"]["title"], "->", len(serie["attributes"]["values"]), "valores")

Código de estado: 200
PVPC -> 24 valores
Precio mercado spot -> 96 valores


In [26]:
# Juntamos las series en una tabla, igual que en la sección 6
tablas = []

for serie in series:
    tabla = pd.DataFrame(serie["attributes"]["values"])
    tabla["serie"] = serie["attributes"]["title"]
    tablas.append(tabla)

df_precios = pd.concat(tablas, ignore_index=True)

# Primeras filas de cada serie
df_precios.groupby("serie").head(8)

,value,percentage,datetime,serie
0,156.72,0.542059,2025-01-15T00:00:00.000+01:00,PVPC
1,146.08,0.544100,2025-01-15T01:00:00.000+01:00,PVPC
2,142.65,0.546029,2025-01-15T02:00:00.000+01:00,PVPC
3,139.52,0.540901,2025-01-15T03:00:00.000+01:00,PVPC
4,139.36,0.541225,2025-01-15T04:00:00.000+01:00,PVPC
5,138.84,0.540297,2025-01-15T05:00:00.000+01:00,PVPC
6,158.77,0.539429,2025-01-15T06:00:00.000+01:00,PVPC
7,194.79,0.537930,2025-01-15T07:00:00.000+01:00,PVPC
24,132.40,0.457941,2025-01-15T00:00:00.000+01:00,Precio mercado spot
25,132.40,1.000000,2025-01-15T00:15:00.000+01:00,Precio mercado spot


### Mismo análisis en otro día: 2 de octubre de 2025

Cambiamos solo la fecha y miramos de nuevo las primeras filas del precio spot.

In [27]:
parametros_precios["start_date"] = "2025-10-02T00:00"
parametros_precios["end_date"] = "2025-10-02T23:59"

respuesta = requests.get(url, params=parametros_precios, timeout=30)
spot = respuesta.json()["included"][1]   # la segunda serie es el precio spot

print(spot["attributes"]["title"])
pd.DataFrame(spot["attributes"]["values"]).head(8)

Precio mercado spot


,value,percentage,datetime
0,105.68,0.429541,2025-10-02T00:00:00.000+02:00
1,105.01,1.000000,2025-10-02T00:15:00.000+02:00
2,103.33,1.000000,2025-10-02T00:30:00.000+02:00
3,103.33,1.000000,2025-10-02T00:45:00.000+02:00
4,102.28,0.429207,2025-10-02T01:00:00.000+02:00
5,100.00,1.000000,2025-10-02T01:15:00.000+02:00
6,99.00,1.000000,2025-10-02T01:30:00.000+02:00
7,98.70,1.000000,2025-10-02T01:45:00.000+02:00


## 8. ¿Cuántos datos devuelve la API en una sola petición?

Probamos rangos de fechas cada vez más largos con datos por hora y miramos el código de estado.

In [28]:
# Fecha de fin para cada prueba (todas empiezan el 1 de enero de 2025)
pruebas = {
    "7 días":  "2025-01-07T23:59",
    "31 días": "2025-01-31T23:59",
    "32 días": "2025-02-01T23:59",
    "1 año":   "2025-12-31T23:59",
}

for nombre, fecha_fin in pruebas.items():
    parametros_prueba = {
        "start_date": "2025-01-01T00:00",
        "end_date": fecha_fin,
        "time_trunc": "hour",
    }
    respuesta = requests.get(url, params=parametros_prueba, timeout=60)
    print(nombre, "-> código", respuesta.status_code)

7 días -> código 200
31 días -> código 200
32 días -> código 400
1 año -> código 400


In [29]:
# ¿Qué dice exactamente la API cuando falla? (respuesta de la última prueba)
respuesta.json()

{'errors': [{'status': '400',
   'title': 'Error Interno',
   'detail': 'Los datos solicitados no están disponibles en este momento. Inténtelo de nuevo más tarde.'}]}